#### Autori: Domenico Lembo, Giuseppe Santucci and Marco Schaerf

[Dipartimento di Ingegneria informatica, automatica e gestionale](https://www.diag.uniroma1.it)

<img src="https://mirrors.creativecommons.org/presskit/buttons/88x31/png/by-nc-sa.eu.png"
     alt="License"
     style="float: left;"
     height="40" width="100" />
This notebook is distributed with license Creative Commons *CC BY-NC-SA*

# Gestione in memoria degli oggetti mutabili
1. Gestione della memoria: stack ed heap
2. Passaggio e gestione parametri di tipo immutabile
3. Passaggio e modifica parametri di tipo lista 
    - 3.1 Esempi di funzioni su liste: modifica elemento di una lista
    - 3.2 Cicli che modificano il numero di elementi in una lista 
        - 3.2.1 Esempio: Pulizia lista
        - 3.2.2 Esempio: Elimina la prima occorrenza
        - 3.2.3 Esempio: Elimina tutte le occorrenze
4. Confronto fra metodi ed operatori sulle liste
    - 4.1 Confronto fra append e +
    - 4.2 Uso di slice per appendere elementi ad una lista
    - 4.3 Approfondimento su `insert()` e `extend()`
    - 4.4 Approfondimento su  `pop()`
    - 4.5 Differenza fra  `l.sort()` e `sorted(l)`
    - 4.6 Copia di liste
    - 4.7 Inizializzazione di una lista

## Gestione della memoria: stack ed heap
Per capire meglio la differenza tra oggetti *immutabili* (come numeri e stringhe) ed oggetti *mutabili* (come liste) è importante avere chiaro come i linguaggi di programmazione ed i sistemi operativi gestiscono la memoria. Quello che presenteremo è una versione **semplificata** di come funziona veramente il gestore della memoria, ma dovrebbe essere sufficiente per il nostro scopo.

Possiamo pensare alla memoria centrale del nostro computer come divisa in varie parti:
* Area di memoria del Codice: Area dove viene memorizzato il codice (istruzioni) del nostro programma
* Area di memoria dei dati: Area dove vengono memorizzati i dati del nostro programma
* Area di lavoro di tipo Stack: Area di lavoro (**privata di ogni programma/funzione**) che contiene i dati *statici* del programma/funzione. Questa zona (di ogni programma/funzione) si chiama generalmente *frame* ed esiste solo per il periodo in cui il programma/funzione è in esecuzione.
* Area di lavoro di tipo Heap: Area di lavoro (**comune a tutti i programmi/funzioni**) che contiene i dati *dinamici* di ogni programma/funzione. Rimane occupata anche dopo la terminazione del programma/funzione se non liberata (*memory garbage*, spesso ripulita dai cosiddetti *garbage collectors*)

In modo riassuntivo possiamo pensare ad una struttura fatta così, con i due casi in cui ci sia un solo programma/funzione in esecuzione o più di uno:

<!--
<a title="By TomV13 [CC BY-SA 3.0 
 (https://creativecommons.org/licenses/by-sa/3.0
)], from Wikimedia Commons" href="https://commons.wikimedia.org/wiki/File:Single_vs_multithreaded_processes.jpg"><img width="512" alt="Single vs multithreaded processes" src="https://upload.wikimedia.org/wikipedia/commons/thumb/e/e3/Single_vs_multithreaded_processes.jpg/512px-Single_vs_multithreaded_processes.jpg"></a>
-->

![Immagine non disponibile](Immagini/Single_vs_multithreaded_processes.jpg)


Il *frame* del programma principale viene normalmente chiamato *global frame*. Il *frame* di ogni programma/funzione viene quindi creato nello stack ed ha, di regola, una dimensione fissa e contiene, oltre ad altre informazioni, **Tutte le variabili** del programma/funzione.

**Ci sono però importanti differenze nel modo in cui variabili di diversi tipi vengono gestite in Python**. In particolare, la differenza fondamentale è tra:
1. Tipi immmutabili (numeri, stringhe, ..): viene memorizzato all'interno del frame sia la variabile che la zona di memoria corrispondente
2. Tipi mutabili (liste, dizionari, ...): viene memorizzato all'interno del frame solo la variabile, mentre la zona di memoria corrispondente si trova nell'heap

Usando il sito web [Python Tutor](http://www.pythontutor.com/), vediamo un [esempio](https://goo.gl/p3XCdG) di esecuzione su questo programma:
```Python
def aggiungi1(l,n):
    l.append(n)

def aggiungi2(l,n):
    l2=l.copy()
    l2.append(n)
    return l2

lista = [2,4,5]
n = 7
aggiungi1(lista,n)
print(lista)
lista2 = aggiungi2(lista,n)
print(lista,lista2)
```

## Passaggio e modifica parametri di tipo immutabile

In [ ]:
## Cosa succede ad a e b ??
def ScambiaVar(x,y):
    z = x
    x = y
    y = z
    
a = 3
b = 4
ScambiaVar(a,b)
print(a,b)  

## Passaggio e modifica parametri di tipo lista 
Siccome le liste sono tipi di dato *mutabili* è possibile modificarle all'interno di una funzione, ottenendo in questo modo un effetto collaterale. Possono essere modificati all'interno della funzione sia i parametri (**più corretto**) che le variabili globali (**da evitare, se possibile**). Vediamo degli esempi (anche su Python Tutor [qui](https://goo.gl/YLBhtN)):

In [ ]:
def raddoppiaUltimo(l): # riceve in ingresso un parametro e lo modifica
    if len(l) > 0:
        l.append(l[-1])

def raddoppiaUltimo2(): # modifica direttamente una variabile globale (NON RACCOMANDATO)
    if len(lista) > 0:
        lista.append(lista[-1])

lista = [2,4,5]
print('Prima della chiamata di funzione:',lista)
raddoppiaUltimo(lista)
print('Dopo la chiamata di funzione:',lista)
raddoppiaUltimo2()
print('Dopo la chiamata della funzione 2:',lista)

Si noti quindi che quando all'interno di una funzione si vuole modificare una variabile globale che si riferisce ad un tipo di dato mutabile **non** si deve dichiarare questa variabile `global`(come avviene per tipi di dato immutabili) 

### Esempi di funzioni su liste: modifica elemento di una lista
Vediamo ora come scrivere una funzione che riceve in ingresso una lista di numeri interi e la modifica raddoppiando tutti gli elementi positivi (anche su Python Tutor [qui](https://goo.gl/KMLG1B)):

In [ ]:
def raddPos(l):
    for i in range(len(l)):
        if l[i] > 0:
            l[i] = 2*l[i]
            
numeri = [3, -2, 1, 6, 0]
raddPos(numeri)
print(numeri)

**Versione NON corretta**

Notate che il ciclo è sulle posizioni, non sugli elementi. Infatti, la versione con in ciclo sugli elementi non modifica la lista (anche su Python Tutor [qui](https://goo.gl/gvBmdf)):

In [ ]:
def raddPos(l):
    for el in l:
        if el > 0:
            el = 2*el
            
numeri = [3, -2, 1, 6, 0]
raddPos(numeri)
print(numeri)

### Cicli che modificano il numero di elementi in una lista: 

#### Esempio: Pulizia lista
Scrivere una funzione che prende in ingresso una lista `l` e rimuove da `l` tutti gli elementi che NON sono stringhe.

**Nota:** per ottenere il tipo di una variabile `x` si usa la funzione `type(x)`. Il tipo stringa è denotato in Python da `str` (altri tipi sono `int`, `float`, `list`, `tuple`, ecc.). Con l'operatore `is` è poi possibile confrontare il tipo restituito da `type()` con il tipo `str`.

In [ ]:
# prima versione - uso del for che itera sugli elementi della lista

#La funzione type() restituisce il tipo dell'elemento e la condizione
#(type(l[i]) is not str) ci dice se l'elemento NON è una stringa.


def rimuoviNonStringhe(l):
    for elem in l:
        if type(elem) is not str:
            l.remove(elem)


lista = [3, 'alla', 'pelle', 9.1, 'casa']

print(lista)
rimuoviNonStringhe(lista)
print(lista)            
            
lista = [3, 5, 'alla', 'pelle', 9.1, 'casa']

print(lista)
rimuoviNonStringhe(lista)
print(lista)

Vediamo l'esecuzione in PythonTutor [qui](https://pythontutor.com/visualize.html#code=%23%20prima%20versione%20-%20uso%20del%20for%20che%20itera%20sugli%20elementi%20della%20lista%0A%0A%23La%20funzione%20type%28%29%20restituisce%20il%20tipo%20dell'elemento%20e%20la%20condizione%0A%23%28type%28l%5Bi%5D%29%20is%20not%20str%29%20ci%20dice%20se%20l'elemento%20NON%20%C3%A8%20una%20stringa.%0A%0A%0Adef%20rimuoviNonStringhe%28l%29%3A%0A%20%20%20%20for%20elem%20in%20l%3A%0A%20%20%20%20%20%20%20%20if%20type%28elem%29%20is%20not%20str%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20l.remove%28elem%29%0A%0A%0Alista%20%3D%20%5B3,%20'alla',%20'pelle',%209.1,%20'casa'%5D%0A%0Aprint%28lista%29%0ArimuoviNonStringhe%28lista%29%0Aprint%28lista%29%20%20%20%20%20%20%20%20%20%20%20%20%0A%20%20%20%20%20%20%20%20%20%20%20%20%0Alista%20%3D%20%5B3,%205,%20'alla',%20'pelle',%209.1,%20'casa'%5D%0A%0Aprint%28lista%29%0ArimuoviNonStringhe%28lista%29%0Aprint%28lista%29&cumulative=false&curInstr=33&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=3&rawInputLstJSON=%5B%5D&textReferences=false). Dopo aver eliminato un elemento in posizione `i`, tutti gli elementi che lo seguono vengono spostati all’indietro di una posizione. All’iterazione successiva il ciclo `for` assegna ad `elem` l’elemento di `l` nella posizione `i+1`, e quindi “salta” al di là dell’elemento che era il successivo prima dell’eliminazione.

Risolviamo facendo una copia della lista ed iterando su questa.

In [ ]:
# seconda versione (corretta) - uso del for che itera su una copia della lista

def rimuoviNonStringhe(l):
    l2=l.copy()
    for elem in l2:
        if type(elem) is not str:
            l.remove(elem)

lista = [3, 5, 'alla', 'pelle', 9.1, 'casa']

print(lista)
rimuoviNonStringhe(lista)
print(lista)

Vedi l'esecuzione su [PythonTutor](https://pythontutor.com/visualize.html#code=def%20rimuoviNonStringhe%28l%29%3A%0A%20%20%20%20l2%3Dl.copy%28%29%0A%20%20%20%20for%20elem%20in%20l2%3A%0A%20%20%20%20%20%20%20%20if%20type%28elem%29%20is%20not%20str%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20l.remove%28elem%29%0A%0Alista%20%3D%20%5B3,%205,%20'alla',%20'pelle',%209.1,%20'casa'%5D%0A%0Aprint%28lista%29%0ArimuoviNonStringhe%28lista%29%0Aprint%28lista%29&cumulative=false&curInstr=0&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=3&rawInputLstJSON=%5B%5D&textReferences=false).

Notiamo inoltre che non è possibile risolvere il problema precedente iterando sugli indici.

In [ ]:
# terza versione (errata) - uso del for che itera sugli indici

def rimuoviNonStringhe2(l):
    for i in range(len(l)):
        # print("la variabile i è pari a ",i)
        # print("la lista è uguale a ", l)
        if type(l[i]) is not str:
            l.remove(l[i])
            
lista = [3, 'alla', 'pelle', 9.1, 'casa']

print(lista)
rimuoviNonStringhe2(lista)
print(lista)

Nella funzione appena vista, in tutti i casi in cui la lista in input presenta almeno un elemento non stringa otteniamo sicuramente un IndexError. Infatti, la funzione range è eseguita una sola volta all'inizio del ciclo, e quindi genera una sequenza di interi che va da 0 a len(l)-1, dove l è la stringa iniziale (nel codice precedente len(l)-1=4). Dato però che l viene "accorciata" durante il ciclo, ad un certo punto il codice prova ad accedere ad una posizione non più esistente generando un IndexError. Vedi esecuzione su PythonTutor [qui](https://pythontutor.com/visualize.html#code=%23%20terza%20versione%20%28errata%29%20-%20uso%20del%20for%20che%20itera%20sugli%20indici%0A%0Adef%20rimuoviNonStringhe2%28l%29%3A%0A%20%20%20%20for%20i%20in%20range%28len%28l%29%29%3A%0A%20%20%20%20%20%20%20%20%23%20print%28%22la%20variabile%20i%20%C3%A8%20pari%20a%20%22,i%29%0A%20%20%20%20%20%20%20%20%23%20print%28%22la%20lista%20%C3%A8%20uguale%20a%20%22,%20l%29%0A%20%20%20%20%20%20%20%20if%20type%28l%5Bi%5D%29%20is%20not%20str%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20l.remove%28l%5Bi%5D%29%0A%20%20%20%20%20%20%20%20%20%20%20%20%0Alista%20%3D%20%5B3,%20'alla',%20'pelle',%209.1,%20'casa'%5D%0A%0Aprint%28lista%29%0ArimuoviNonStringhe2%28lista%29%0Aprint%28lista%29&cumulative=false&curInstr=0&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=3&rawInputLstJSON=%5B%5D&textReferences=false)

La soluzione consiste in questo caso nel **bloccare l’avanzamento dell’indice in caso di eliminazione**, in modo che all’iterazione successiva si consideri nuovamente la posizione appena analizzata (che contiene un nuovo elemento). Per fare questo è necessario usare un ciclo `while`.

In [ ]:
# quarta versione (corretta) - con ciclo while
#
#Notate l'uso del while al posto del for per evitare problemi con
#la modifica della lista mentre la scorriamo

def rimuoviNonStringhe3(l):
    i = 0
    while i < len(l):
        if type(l[i]) is not str:
            l.remove(l[i])
        else:
            i = i+1 
            # i si incrementa solo se non è stata eliminato nulla.

lista = [3, 5, 'alla', 'pelle', 9.1, 'casa']

print(lista)
rimuoviNonStringhe3(lista)
print(lista)

#### Esempio: elimina la prima occorrenza
scrivere una funzione python che ricevendo in ingresso una lista `l` e un elemento `elem` da rimuovere, ne rimuove la prima occorrenza (se compare nella lista) restituendo True, altrimenti restituisce False


In [ ]:
#rimuove il primo elemento dalla lista. Restituisce True se la rimozione ha successo
def rimuovi(l,elem):
    if elem in l:
        l.remove(elem)
        return True
    else:
        return False

k=['nel','mezzo','cammino','nel']

print(k)
print("rimuovo 'casa'",rimuovi(k,'casa'))
print("rimuovo 'nel'",rimuovi(k,'nel'))
print(k)

#### Esempio: elimina tutte le occorrenze
Scrivere una funzione Python che riceve in ingresso una lista `l` ed un elemento `el` e rimuove (se presente) tutte le occorrenze di `el` in `l`, restituendo True. Se l'elemento non è presente in `l`, restituisce False.

In [ ]:
#rimuove TUTTE le occorrenze di un elemento dalla lista.
#Restituisce True se c'è almeno un elemento da rimuovere

def rimuoviTutti(l,el):
    rimosso =  el in l
    while el in l:
        l.remove(el)       
    return rimosso

k=['nel','mezzo','cammino','mezzo','giorno']

print(k)
print("rimuovo 'mezzo'",rimuoviTutti(k,'mezzo'))
print(k)
print("rimuovo 'nel'",rimuoviTutti(k,'nel'))
print(k)
print("rimuovo 'vita'",rimuoviTutti(k,'vita'))
print(k)

La seguente versione che usa il `for` è invece errata.

In questa soluzione infatti si effettua un ciclo for sugli elementi della lista e se l'elemento è quello da eliminare allora lo elimina. Questa soluzione crea però dei problemi in alcuni casi, dovuti al fatto che stiamo facendo un ciclo for su una lista che allo stesso tempo stiamo modificando. Poiché il ciclo for, per ottimizzare il codice, calcola solo all'inizio la sequenza su cui deve fare il ciclo e **non** si aspetta che venga modificata. Anche se iteriamo sugli elementi e non sugli indici, se ad una iterazione è stato analizzato l'elemento in posizione i, all'iterazione successiva sarà analizzato l'elemento in posizione i+1 (se esiste), e se nell'iterazione la lista è stata modificata e l'elemento in posizione i+1 è cambiato, il ciclo analizzerà il nuovo elemento (se non esiste più, il ciclo si ferma). Non sempre quindi tutti gli elementi della lista vengono effettivamente analizzati. 

Notate che nel caso di sotto, la prima eliminazione funziona correttamente, ma l'ultima lascia una copia dell'elemento.

In [ ]:
def rimuoviTuttiErr(l,el):
    rimosso =  el in l
    for elem in l:
        if elem == el:
            l.remove(elem)
    return rimosso


k=['nel','mezzo','giorno','giorno','cammino','giorno']

print(k)
print("rimuovo 'mezzo'",rimuoviTuttiErr(k,'mezzo'))
print(k)
print("rimuovo 'vita'",rimuoviTuttiErr(k,'vita'))
print(k)
print("rimuovo 'giorno'",rimuoviTuttiErr(k,'giorno'))
print(k)



## Confronto fra metodi ed operatori sulle liste

### Confronto fra append e +
I seguenti frammenti di codice possono sembrare equivalenti. Hanno in pratica lo stesso effetto ma la gestione della memoria è diversa 

In [ ]:
l = [1,2,3]
l.append(12)
print(l)

print()

l = [1,2,3]
l = l + [12] #notare l'uso di una lista contenente l'elemento 12
print(l)

Possiamo usare la funzione `id()` che ci restituisce un identificativo univoco per l'oggetto. Ogni oggetto ha un id differente. Intuitivamente, corrisponde all'indirizzo di memoria occupata dall'oggetto.

In [ ]:
l = [1,2,3]
print(id(l))
l.append(4)
print(l)
print(id(l))

print()

l = [1,2,3]
print(id(l))
l = l + [4] #notare l'uso di una lista contenente l'elemento 4
print(l)
print(id(l))

Questo è particolarmente rilevante quando andiamo a scrivere delle funzioni. Esegui in [Python Tutor](https://pythontutor.com/live.html#code=def%20raddoppiaUltimo%28ls%29%3A%20%23%20riceve%20in%20ingresso%20un%20parametro%20e%20lo%20modifica%0A%20%20%20%20if%20len%28ls%29%20%3E%200%3A%0A%20%20%20%20%20%20%20%20print%28%22Identificativo%20di%20ls%20prima%20della%20modifica%22,%20id%28ls%29%29%0A%20%20%20%20%20%20%20%20%23l.append%28l%5B-1%5D%29%0A%20%20%20%20%20%20%20%20ls%20%3D%20ls%20%2B%20%5Bls%5B-1%5D%5D%0A%20%20%20%20%20%20%20%20print%28%22Identificativo%20di%20ls%20dopo%20la%20modifica%22,%20id%28ls%29%29%0A%20%20%20%20%20%20%20%20%0A%0Al%20%3D%20%5B2,4,5%5D%0Aprint%28'Prima%20della%20chiamata%20di%20funzione%3A',l%29%0Aprint%28'Identificativo%20di%20l',%20id%28l%29%29%0AraddoppiaUltimo%28l%29%0Aprint%28'Dopo%20la%20chiamata%20di%20funzione%3A',l%29%0Aprint%28'Identificativo%20di%20l',%20id%28l%29%29&cumulative=false&curInstr=0&heapPrimitives=nevernest&mode=display&origin=opt-live.js&py=3&rawInputLstJSON=%5B%5D&textReferences=false)

In [ ]:
# In questa nuova versione di raddoppiaUltimo sostituiamo
# la append con una concatenazione. Come possiamo vedere
# questa soluzione è scorretta: la lista in input non viene 
# modificata

def raddoppiaUltimo(ls): # riceve in ingresso un parametro e lo modifica
    if len(ls) > 0:
        print("Identificativo di ls prima della modifica:", id(ls))
        #ls.append(ls[-1])
        ls = ls + [ls[-1]]  # sostituendo il rigo precedente con questo la modifica 
                            # non si ripercuote sulla variabile l del programma
        print("Identificativo di ls dopo la modifica:", id(ls))
        

l = [2,4,5]
print('l prima della chiamata di funzione:',l)
print('Identificativo di l:', id(l))
raddoppiaUltimo(l)
print('l dopo la chiamata di funzione:',l)
print('Identificativo di l:', id(l))

### Uso di slice per appendere elementi ad una lista

Prima di tutto notiamo quanto segue
 
`l=[1,2,3,4]`

`l[4]`

<span style="color:red">Traceback (most recent call last):</span> <br>
 <span style="color:red"> File "<pyshell#155>", line 1, in <module\></span><br>
 <span style="color:red">   l[4]</span><br>
<span style="color:red">IndexError: list index out of range</span>

`l[4:4]`

[ ]
 
Mentre l’operazione di indicizzazione (`l[4]`) tenta di accedere ad una posizione che non esiste in `l`, l’operazione di slicing in cui i due indici `i` sono uguali (nell’esempio `l[4:4]`) individua la sottolista di `l` che inizia alla posizione `i`, ma non comprende questa posizione. In questi casi la sottolista è ovviamente vuota (ma va comunque considerata una sottolista di `l` posta “appena prima della posizione” `i` ).

Possiamo quindi sempre appendere un elemento ad una lista `l` modificando la sua sottolista (vuota) che otteniamo tramite `l[len(l):len(l)]`.


In [ ]:
#Ad esempio

l=[1,2,3,4]

l[len(l):len(l)] = [5] 
print(l)

**Nota:** in realtà questo meccanismo funziona anche con `l[x:len(l)]`, per ogni `x >= len(l)` 

### Approfondimento su `insert()` e `extend()`

`l.insert(i,elem)` inserisce l’elemento `elem` nella lista `l` in posizione `i` (se `i>=0`, `i` sarà in effetti la posizione di `elem` dopo l’inserimento; se `i<0`, la posizione dopo l'inserimento sarà `i-1`). Tutti gli elementi che si trovavano in posizione `0>=k>=i` prima dell’invocazione della `insert()` si troveranno in posizione `k+1` dopo l’invocazione della `insert()`. Si noti che `l.insert(0,elem)` inserisce `elem` in testa alla lista `l`, e `l.insert(len(l),elem))` equivale a `l.append(elem)`. 

Si noti anche che le seguenti istruzioni modificano `l` nello stesso modo:
 
`l.insert(i, elem)`<br> 
`l[i:i] = [elem]`
 
In particolare, la seconda espressione sostituisce la sottolista `[]` di `l` che si trova prima dell’elemento `i` con la lista `[elem]` (notate che è anche possibile scrivere `l[i:j] = [elem]`, con `i>j` ed avere il medesimo effetto).

Il metodo `extend()` nell’invocazione `l.extend(L)` estende la lista aggiungendovi in coda tutti gli elementi della lista `L`; equivale a `l[len(l):len(l)] = L`.

In [ ]:
#Ad esempio

l=[1,2,4]
l[2:2]=[3] # equivale a l.insert(2,3)
print(l)

print()

s=[5,6]

l[len(l):len(l)] = s # equivale a l.extend(s)
print(l)

### Approfondimento su  `pop()`

`pop(i)` rimuove l'elemento di indice `i` e lo restituisce come risultato dell'operazione. Si noti che entrambe le seguenti istruzioni modificano `l` nello stesso modo
 
`l.pop(i)`<br>
`l[i:i+1] = [] `

anche se `l.pop(i)` in più fornisce come risultato anche l’elemento eliminato.
 
Se non è specificato alcun indice, `l.pop()` rimuove l'ultimo elemento della lista e lo restituisce come risultato dell’operazione. 


### Differenza fra  `l.sort()` e `sorted(l)`

`l.sort()` ordina gli elementi della lista `l` in ordine crescente, modificando `l`. Non restituisce alcun risultato. I suoi effetti sono visibili solo sulla lista `l`

`sorted(l)` è una **funzione** che restituisce una **nuova lista** i cui elementi sono uguali a quelli della lista `l` ma ordinati in ordine crescente. `l` non viene modificata. L'effetto dell'ordinamento è visibile solo nella lista restituita.

In [ ]:
l = [3,2,4,5,1]
print(l.sort())
print(l)

In [ ]:
l = [3,2,4,5,1]
print(sorted(l))
print(l)

### Copia di liste

Le due seguenti istruzioni sono equivalenti (entrambi consentono di fare una copia della lista).

`s=l.copy()`<br>
`s=l[:]`

In [ ]:
l=[1,2,3]
s = l.copy()
print(s)
print("l ed s sono uguali?",l==s)
print("l ed s sono la stessa lista?",l is s)

print()

l=[1,2,3]
s = l[:]
print(s)
print("l ed s sono uguali?",l==s)
print("l ed s sono la stessa lista?",l is s)

### Inizializzazione di una lista
Inizializzare una lista vuol dire creare una lista di una certa lunghezza i cui elementi siano tutti impostati ad un valore iniziale (in genere un elemento non significativo e destinato ad essere modificato). Questo modo di procedere può essere particolarmente comodo quando dobbiamo manipolare liste di dimensione nota a priori (e che non viene modificata), in modo da allocare inizialmente tutte le componenti della lista, e successivamente popolarle con elementi significativi. Per mostrare un caso in cui è importante avere delle liste già inizializzate vediamo una funzione che somma due vettori

In [ ]:
## funzione che riceve in ingresso due liste di pari dimensioni contenententi numeri interi
## e rappresentanti due vettori, e ne restituisce la lista rappresentante il vettore somma

from tester import tester_fun

def sommaVettori(v1,v2):
    length=len(v1)
    l=[0] * length
    for j in range(length):
        l[j]=v1[j]+v2[j]
    return l
        
tester_fun(sommaVettori,[[3,5,8],[2,3,4]],[5,8,12])
tester_fun(sommaVettori,[[1,1,1,1],[2,2,2,2]],[3,3,3,3])

In Python ci sono vari modi per inizializzare una lista (oltre quello visto in precedenza). In particolare è possibile usare la cosiddetta *list comprehension*

In [ ]:
# creazione di una lista di 10 elementi tutti 0
l1 = [0]*10
print('l1=',l1)

# in alternativa si può usare questa forma (i.e., list comprehension)
l2 = [0 for i in range(10)]
print('l2=',l2)

# questa seconda forma è più flessibile, per esempio
l3 = [i for i in range(10)]
print('l3=',l3)

# la forma precedente è del tutto equivalente al codice seguente
l4 = []
for i in range(10):
    l4.append(i)
print('l4=',l4)    

Vediamo ora come inizializzare una **lista di liste**, tutte inizialmente vuote 

In [ ]:
# Creazione di una lista di liste, tutte della stessa lunghezza. 
# Attenzione: il codice che segue crea una sola
# lista di 3 zeri e 4 riferimenti a questa stessa lista
l5 = [[0]*3]*4
print('l5=',l5)

l5[1][1]=1
print('l5=',l5)

# in alternativa si può usare il seguente codice
# Attenzione: in questo caso vengono prodotte 4 liste "diverse" 
# (cioè 4 copie della stessa lista) di 3 zeri 
l6 = [[0 for i in range(3)] for i in range(4)]
print('l6=',l6)

l6[1][1]=1
print('l6=',l6)

# stesso codice di sopra ma con l'uso di due variabili/parametri
m=4   #numero di sottoliste   
n=3   #nunero di elementi in ciascuna sottolista
l7 = [[0 for i in range(n)] for i in range(m)]
print('l7=',l7)

#len(l7) --> numero di sottoliste
#len(l7[0]) ---> nunero di elementi in ciascuna sottolista

Come evidenziato nei commenti precedenti, nel caso di inizializzazione di liste di liste l'uso della list comprehension non produce un risultato equivalente all'uso della "moltiplicazione" di una lista per un intero

Per convincersene, si guardi anche <a href="https://pythontutor.com/visualize.html#code=lista1%3D%5B%5B%5D%20for%20i%20in%20range%283%29%5D%0A%0Alista2%3D%5B%5B%5D%5D*3%0A%0Alista3%20%3D%20%5B%5B0%5D*3%5D*4%0A%0Alista4%20%3D%20%5B%5B0%20for%20i%20in%20range%283%29%5D%20for%20i%20in%20range%284%29%5D%0A%0Aprint%28lista3%29%0Aprint%28lista4%29%0A%0Alista3%5B0%5D%5B1%5D%3D1%0A%0Alista4%5B0%5D%5B1%5D%3D1%0A%0Aprint%28lista3%29%0Aprint%28lista4%29&cumulative=false&curInstr=0&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=3&rawInputLstJSON=%5B%5D&textReferences=false">questo codice sul Python Tutor</a>

Si sottolinea infine che, come abbiamo visto, la list comprehension è molto comoda per scrivere codice in modo compatto, ma non aggiunge potere espressivo al linguaggio. Ad esempio, se dobbiamo trasformare una lista di stringhe rappresentati interi (ad esempio acqusiti da riga di comando o letti da un file di testo, come vedremo nelle prossime lezioni), è possibile usare una list comprehension per ottenere un codice Python più compatto.

In [ ]:
listaStringhe=['1','2','10','54','66']

listaInteri=[int(s) for s in listaStringhe]

print(sum(listaInteri))

## è equivalente a

listaInteri=[]
for s in listaStringhe:
    listaInteri.append(int(s))

print(sum(listaInteri))

### Esercizi
Completate questi esercizi prima di cominciare il prossimo argomento

### Esercizio 1: 
Scrivere una funzione Python che riceve in input una lista di stringhe e restituisce la lunghezza della stringa più lunga. 

In [ ]:
from tester import tester_fun

def stringaMax(l):
    """MODIFICARE IL CONTENUTO DI QUESTA FUNZIONE PER SVOLGERE L'ESERCIZIO"""  

tester_fun(stringaMax,[['casa','pippo', 'paperopoli']],10)
tester_fun(stringaMax,[['ciao','a', '','io']],4)
tester_fun(stringaMax,[['mario','maria','mario']],5)
tester_fun(stringaMax,[['a','b','c']],1)
tester_fun(stringaMax,[['abracadabra']],11)

### Esercizio 2:
scrivere una funzione Python che riceve in input una lista di interi e restituisce la posizione dell'ultimo elemento positivo (>= 0). Se nessun elemento è positivo deve restituire -1

In [ ]:
from tester import tester_fun

def ultimoPos(l):
    """MODIFICARE IL CONTENUTO DI QUESTA FUNZIONE PER SVOLGERE L'ESERCIZIO"""

tester_fun(ultimoPos,[[5, 6, 9, 0, -2]],3)
tester_fun(ultimoPos,[[-5, -6, -9, -2]],-1)
tester_fun(ultimoPos,[[-10, 6, 9, 0, 2]],4)
tester_fun(ultimoPos,[[-5, 3]],1)
tester_fun(ultimoPos,[[5, -6, 9, 0, 2, -12, 13]],6)

### Esercizio 3:
Scrivere una funzione python che ricevendo in ingresso una lista `l` e una stringa `s` inserisce la stringa `s` in `l` in modo da mantenere le stringhe ordinate per lunghezza

In [ ]:
from tester import tester_fun

#inserisce una stringa in una lista mantenendo l'ordinamento sulla
#lunghezza delle stringhe

def inserisciStringa(l,s):
    """MODIFICARE IL CONTENUTO DI QUESTA FUNZIONE PER SVOLGERE L'ESERCIZIO"""

tester_fun(inserisciStringa,[['nel','mezzo','cammino'],'casa'],['nel','casa','mezzo','cammino'])
tester_fun(inserisciStringa,[['nel','casa','mezzo','cammino'],'casa'],['nel','casa','casa','mezzo','cammino'])
tester_fun(inserisciStringa,[['nel','casa','casa','mezzo','cammino'],'vita'],['nel','casa','casa','vita','mezzo','cammino'])
tester_fun(inserisciStringa,[['nel','casa','casa','mezzo','cammino'],'nostra'],['nel','casa','casa','mezzo','nostra','cammino'])
tester_fun(inserisciStringa,[['nel','casa','mezzo','nostra','cammino'],'mi'],['mi','nel','casa','mezzo','nostra','cammino'])

### Esercizio 4:
Scrivere una funzione che prende in ingresso una lista `l` di elementi disomogenei e calcola la somma di tutti gli elementi interi.

In [ ]:
from tester import tester_fun

def somma(l):
    """MODIFICARE IL CONTENUTO DI QUESTA FUNZIONE PER SVOLGERE L'ESERCIZIO"""

tester_fun(somma,[[3, 5, 6.1, -1, 'casa',  11]],18)
tester_fun(somma,[[3, 'ciao', 6.1, -1, 'casa',  11]],13)
tester_fun(somma,[[3, 'ciao', 'roma', -1, 'casa',  11]],13)
tester_fun(somma,[[3, -1, 11]],13)
tester_fun(somma,[[5, 'ciao', -1, 'casa',  13]],17)

### Esercizio 5:
Scrivere una funzione che prende in input una lista di stringhe `l` ed un intero `i` ed elimina da `l` tutte le stringhe che hanno lunghezza maggiore di `i `

In [ ]:
from tester import tester_fun

def eliminaStringhe(l,i) :
    """MODIFICARE IL CONTENUTO DI QUESTA FUNZIONE PER SVOLGERE L'ESERCIZIO"""

tester_fun(eliminaStringhe,[["mamma","sì","ciao","pippo"],2],["sì"])
tester_fun(eliminaStringhe,[["mamma","sì","ciao","pippo"],4],["sì", "ciao"])
tester_fun(eliminaStringhe,[["mamma","sì","ciao","pippo"],5],["mamma","sì","ciao","pippo"])
tester_fun(eliminaStringhe,[["mamma","sì","ciao","pippo"],0],[])
tester_fun(eliminaStringhe,[["assolo","io"],4],["io"])

### Esercizio 6:
Scrivere un funzione che prende in input 2 liste di stringhe ordinate per lunghezza e crea una nuova lista dalla fusione (merge) delle 2 liste preservando l'ordinamento per lunghezza. Nel caso di stringhe della stessa lunghezza, scegliere prima le stringhe della lista `l2`.

In [ ]:
from tester import tester_fun
    
def merge(l1,l2):
    """MODIFICARE IL CONTENUTO DI QUESTA FUNZIONE PER SVOLGERE L'ESERCIZIO"""

tester_fun(merge,[['a', 'b','abc','abbcc'],['','aa','bb','cccc','ccccccc']],['','a','b','aa','bb','abc','cccc','abbcc','ccccccc'])
tester_fun(merge,[['aa','bb'],['cccc','ccccccc']],['aa','bb','cccc','ccccccc'])
tester_fun(merge,[["sì","ciao","mamma","pippo"],["io","assolo"]],["io","sì","ciao","mamma","pippo","assolo"])
tester_fun(merge,[['', 'a','c'],['dd']],['', 'a','c','dd'])
tester_fun(merge,[['alfa','omega'],['beta','gamma']],['beta','alfa','gamma','omega']) 
